# 第三章：数据建模与批量导入

## 学习目标

- 掌握图数据建模思路（实体→节点，关系→边）
- 使用 Python driver 的事务模式
- 用 UNWIND 批量导入数据
- 创建约束和索引保证数据质量
- 对比逐条写入 vs 批量写入的性能差异

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)

driver.verify_connectivity()
print("✓ Neo4j 连接成功")

In [ ]:
# 清空数据库，确保从零开始
driver.execute_query("MATCH (n) DETACH DELETE n")
print("✓ 数据库已清空")

## 1. 图数据建模思路

我们用一个**学术论文**领域来学习建模。需求是：论文有作者，属于某个研究主题，论文之间有引用关系。

### 建模三步走

1. **识别实体 → 节点标签**
   - 论文 → `Paper`
   - 作者 → `Author`
   - 主题 → `Topic`

2. **识别关系 → 关系类型**
   - 作者写了论文 → `(Author)-[:AUTHORED]->(Paper)`
   - 论文属于主题 → `(Paper)-[:BELONGS_TO]->(Topic)`
   - 论文引用论文 → `(Paper)-[:CITES]->(Paper)`

3. **确定属性 → 键值对**
   - Paper: `title`, `year`, `venue`
   - Author: `name`, `affiliation`
   - Topic: `name`

### 对比：关系型 vs 图

同一份数据在关系型数据库中需要 **至少 5 张表**：

| 表名 | 字段 | 说明 |
|---|---|---|
| `papers` | id, title, year, venue | 论文表 |
| `authors` | id, name, affiliation | 作者表 |
| `topics` | id, name | 主题表 |
| `paper_authors` | paper_id, author_id | 多对多关联表 |
| `paper_topics` | paper_id, topic_id | 多对多关联表 |
| `citations` | citing_id, cited_id | 引用关联表 |

查询“某作者所有论文的引用链”需要 **多重 JOIN**，SQL 写起来非常繁琐。

而在图数据库中，同样的查询只需：

```cypher
MATCH (a:Author {name: '张三'})-[:AUTHORED]->(p:Paper)-[:CITES*1..3]->(cited:Paper)
RETURN p.title, cited.title
```

关系就是**一等公民**——不需要中间表，不需要 JOIN。

## 2. Python Driver 事务模式

第一章中我们用 `driver.execute_query()` 一行代码搞定读写。这是**简写模式**，适合简单场景。

当你需要**更精细的控制**（比如在一个事务里执行多个操作、区分读写路由），就要用**显式事务模式**。

In [ ]:
# 显式事务：写入
def create_author(tx, name, affiliation):
    tx.run(
        "MERGE (a:Author {name: $name}) SET a.affiliation = $affiliation",
        name=name, affiliation=affiliation,
    )

with driver.session() as session:
    session.execute_write(create_author, "张三", "北京大学")
    session.execute_write(create_author, "李四", "清华大学")
    print("✓ 作者创建完成")

In [ ]:
# 显式事务：读取
def get_authors(tx):
    result = tx.run("MATCH (a:Author) RETURN a.name AS name, a.affiliation AS affiliation")
    return [record.data() for record in result]

with driver.session() as session:
    authors = session.execute_read(get_authors)
    for a in authors:
        print(f"  {a['name']} - {a['affiliation']}")

### 刚才发生了什么？

1. 定义了一个**事务函数** `create_author(tx, ...)`，它接收一个事务对象 `tx` 和业务参数
2. `session.execute_write()` 把函数交给 Neo4j 执行，框架负责：
   - 开启事务
   - 执行函数
   - 提交事务（出错则回滚）
   - **遇到瞬时错误自动重试**（比如网络抖动、主从切换）
3. `execute_read()` 同理，但标记为只读事务——在集群环境中会路由到**只读副本**，减轻主节点压力

### 对比：两种事务方式

| | `driver.execute_query()` | `session.execute_write/read()` |
|---|---|---|
| 适合场景 | 简单的一次性操作 | 需要精细控制的业务逻辑 |
| 事务管理 | 自动（一条语句一个事务） | 显式（可在一个事务内多步操作） |
| 读写路由 | 通过 `routing_` 参数控制 | `execute_read` 自动路由到只读副本 |
| 自动重试 | 支持 | 支持 |
| 代码量 | 少 | 多一些，但更灵活 |

In [ ]:
# 清理，为下一节做准备
driver.execute_query("MATCH (n) DETACH DELETE n")
print("✓ 数据已清理")

## 3. 对比：逐条写入 vs 批量写入

在实际项目中，数据量往往成百上千条。写入方式的选择直接影响性能。

In [ ]:
import time

# 准备测试数据
test_papers = [{"title": f"论文_{i}", "year": 2020 + i % 5} for i in range(100)]

In [ ]:
# ❌ 逐条写入（慢）
driver.execute_query("MATCH (n:TestPaper) DETACH DELETE n")

start = time.time()
for paper in test_papers:
    driver.execute_query(
        "CREATE (p:TestPaper {title: $title, year: $year})",
        title=paper["title"], year=paper["year"],
    )
elapsed_slow = time.time() - start
print(f"逐条写入 {len(test_papers)} 条: {elapsed_slow:.2f} 秒")

In [ ]:
# ✅ UNWIND 批量写入（快）
driver.execute_query("MATCH (n:TestPaper) DETACH DELETE n")

start = time.time()
driver.execute_query(
    "UNWIND $papers AS paper CREATE (p:TestPaper {title: paper.title, year: paper.year})",
    papers=test_papers,
)
elapsed_fast = time.time() - start
print(f"批量写入 {len(test_papers)} 条: {elapsed_fast:.2f} 秒")
print(f"速度提升: {elapsed_slow / elapsed_fast:.1f}x")

# 清理
driver.execute_query("MATCH (n:TestPaper) DETACH DELETE n")

### 刚才发生了什么？

**逐条写入**：每条数据单独发一次网络请求、开一个事务，100 条 = 100 次往返。

**UNWIND 批量写入**：把整个列表作为参数传给 Neo4j，`UNWIND` 在**一个事务**内把列表展开逐行处理。网络往返只有 **1 次**。

```cypher
UNWIND $papers AS paper
CREATE (p:TestPaper {title: paper.title, year: paper.year})
```

`UNWIND` 相当于 Cypher 中的 `for` 循环——接收一个列表参数，依次将每个元素绑定到变量 `paper`，然后执行后面的语句。

> **经验法则**：数据量超过 10 条就该用 UNWIND。数据量特别大（>10 万）时，建议分批（每批 5000–10000 条）。

## 4. 实战：构建学术论文知识图谱

现在用 UNWIND 批量导入的方式，构建一个完整的学术论文知识图谱。

In [ ]:
# 准备数据
papers_data = [
    {"title": "Attention Is All You Need", "year": 2017, "venue": "NeurIPS"},
    {"title": "BERT: Pre-training of Deep Bidirectional Transformers", "year": 2018, "venue": "NAACL"},
    {"title": "GPT-3: Language Models are Few-Shot Learners", "year": 2020, "venue": "NeurIPS"},
    {"title": "LLaMA: Open and Efficient Foundation Language Models", "year": 2023, "venue": "arXiv"},
    {"title": "RAG: Retrieval-Augmented Generation", "year": 2020, "venue": "NeurIPS"},
]

authors_data = [
    {"name": "Ashish Vaswani", "affiliation": "Google"},
    {"name": "Jacob Devlin", "affiliation": "Google"},
    {"name": "Tom Brown", "affiliation": "OpenAI"},
    {"name": "Hugo Touvron", "affiliation": "Meta"},
    {"name": "Patrick Lewis", "affiliation": "Meta"},
]

# 论文-作者映射
authorship = [
    {"paper": "Attention Is All You Need", "author": "Ashish Vaswani"},
    {"paper": "BERT: Pre-training of Deep Bidirectional Transformers", "author": "Jacob Devlin"},
    {"paper": "GPT-3: Language Models are Few-Shot Learners", "author": "Tom Brown"},
    {"paper": "LLaMA: Open and Efficient Foundation Language Models", "author": "Hugo Touvron"},
    {"paper": "RAG: Retrieval-Augmented Generation", "author": "Patrick Lewis"},
]

# 引用关系
citations = [
    {"from": "BERT: Pre-training of Deep Bidirectional Transformers", "to": "Attention Is All You Need"},
    {"from": "GPT-3: Language Models are Few-Shot Learners", "to": "Attention Is All You Need"},
    {"from": "LLaMA: Open and Efficient Foundation Language Models", "to": "Attention Is All You Need"},
    {"from": "RAG: Retrieval-Augmented Generation", "to": "GPT-3: Language Models are Few-Shot Learners"},
]

# 论文-主题映射
topics_data = [
    {"paper": "Attention Is All You Need", "topic": "Transformer"},
    {"paper": "BERT: Pre-training of Deep Bidirectional Transformers", "topic": "NLP"},
    {"paper": "GPT-3: Language Models are Few-Shot Learners", "topic": "LLM"},
    {"paper": "LLaMA: Open and Efficient Foundation Language Models", "topic": "LLM"},
    {"paper": "RAG: Retrieval-Augmented Generation", "topic": "RAG"},
]

print(f"准备数据: {len(papers_data)} 篇论文, {len(authors_data)} 位作者, {len(citations)} 条引用, {len(topics_data)} 个主题关系")

In [ ]:
# 批量导入论文
driver.execute_query(
    "UNWIND $papers AS p MERGE (paper:Paper {title: p.title}) SET paper.year = p.year, paper.venue = p.venue",
    papers=papers_data,
)
print(f"✓ 导入 {len(papers_data)} 篇论文")

In [ ]:
# 批量导入作者
driver.execute_query(
    "UNWIND $authors AS a MERGE (author:Author {name: a.name}) SET author.affiliation = a.affiliation",
    authors=authors_data,
)
print(f"✓ 导入 {len(authors_data)} 位作者")

In [ ]:
# 批量创建 AUTHORED 关系
driver.execute_query("""
    UNWIND $mappings AS m
    MATCH (p:Paper {title: m.paper})
    MATCH (a:Author {name: m.author})
    MERGE (a)-[:AUTHORED]->(p)
""", mappings=authorship)
print(f"✓ 创建 {len(authorship)} 个 AUTHORED 关系")

In [ ]:
# 批量创建引用关系
driver.execute_query("""
    UNWIND $cites AS c
    MATCH (from_paper:Paper {title: c.from})
    MATCH (to_paper:Paper {title: c.to})
    MERGE (from_paper)-[:CITES]->(to_paper)
""", cites=citations)
print(f"✓ 创建 {len(citations)} 个 CITES 关系")

In [ ]:
# 批量创建主题关系
driver.execute_query("""
    UNWIND $topics AS t
    MATCH (p:Paper {title: t.paper})
    MERGE (topic:Topic {name: t.topic})
    MERGE (p)-[:BELONGS_TO]->(topic)
""", topics=topics_data)
print("✓ 创建主题关系")

### 刚才发生了什么？

我们用 5 条 UNWIND 语句完成了整个知识图谱的构建：

1. **导入节点**：先用 `MERGE` 创建 Paper 和 Author 节点（`MERGE` 保证不重复）
2. **创建关系**：先 `MATCH` 找到已有节点，再 `MERGE` 创建关系

**为什么关系要分开导入？**

节点必须先存在，关系才能连接。如果把节点创建和关系创建放在一条 UNWIND 里，可能因为节点还没创建就去匹配而失败。

**导入顺序**：
1. 先导入没有依赖的节点（Paper、Author）
2. 再创建节点间的关系（AUTHORED、CITES、BELONGS_TO）
3. Topic 节点可以在创建关系时顺带 MERGE 出来

### 验证：查询知识图谱

In [ ]:
# 查询：被引用最多的论文
print("被引用最多的论文：")
records, _, _ = driver.execute_query("""
    MATCH (p:Paper)<-[:CITES]-(citing:Paper)
    RETURN p.title AS title, count(citing) AS cite_count
    ORDER BY cite_count DESC
""")
for r in records:
    print(f"  {r['title']}: {r['cite_count']} 次")

In [ ]:
# 查询：某个主题下的所有论文
print("LLM 主题的论文：")
records, _, _ = driver.execute_query("""
    MATCH (p:Paper)-[:BELONGS_TO]->(t:Topic {name: 'LLM'})
    RETURN p.title AS title, p.year AS year
    ORDER BY p.year
""")
for r in records:
    print(f"  [{r['year']}] {r['title']}")

In [ ]:
# 查询：论文引用链
print("从 RAG 出发的引用链：")
records, _, _ = driver.execute_query("""
    MATCH path = (start:Paper {title: 'RAG: Retrieval-Augmented Generation'})-[:CITES*1..3]->(cited:Paper)
    RETURN [n IN nodes(path) | n.title] AS chain
""")
for r in records:
    chain = r['chain']
    print(f"  {' → '.join(chain)}")

### 刚才发生了什么？

三个查询展示了图数据库的核心优势：

1. **聚合查询**：`count()` + `ORDER BY` 找出被引最多的论文
2. **模式匹配**：通过 `BELONGS_TO` 关系直接找到某主题下的所有论文——不需要 JOIN
3. **路径查询**：`[:CITES*1..3]` 沿引用关系走 1 到 3 步，找出引用链。这在关系型数据库中需要递归 CTE 或多重 JOIN，写起来非常痛苦

## 5. 约束与索引

数据导入完成了，但还缺少**数据质量保障**。如果有人误操作插入了重复的论文怎么办？如果查询很慢怎么办？

In [ ]:
# 创建唯一约束（自动创建索引）
driver.execute_query(
    "CREATE CONSTRAINT paper_title IF NOT EXISTS FOR (p:Paper) REQUIRE p.title IS UNIQUE"
)
print("✓ Paper.title 唯一约束创建成功")

driver.execute_query(
    "CREATE CONSTRAINT author_name IF NOT EXISTS FOR (a:Author) REQUIRE a.name IS UNIQUE"
)
print("✓ Author.name 唯一约束创建成功")

In [ ]:
# 创建索引（加速非唯一属性的查询）
driver.execute_query(
    "CREATE INDEX paper_year IF NOT EXISTS FOR (p:Paper) ON (p.year)"
)
print("✓ Paper.year 索引创建成功")

In [ ]:
# 查看所有约束和索引
records, _, _ = driver.execute_query("SHOW CONSTRAINTS")
print("约束:")
for r in records:
    print(f"  {r['name']}: {r['type']}")

records, _, _ = driver.execute_query("SHOW INDEXES")
print("\n索引:")
for r in records:
    print(f"  {r['name']}: {r['type']} on {r.get('labelsOrTypes', '?')}")

### 刚才发生了什么？

**唯一约束（UNIQUE Constraint）**：
- 保证同一标签下某属性的值不重复
- 例如 `Paper.title` 加了唯一约束后，尝试插入重复标题的论文会报错
- 创建唯一约束时 Neo4j **自动创建对应的索引**，不需要再手动建

**索引（Index）**：
- 加速 `MATCH (p:Paper {year: 2020})` 这类按属性查找的操作
- 没有索引时 Neo4j 要全量扫描所有 Paper 节点，有索引后直接定位

**`IF NOT EXISTS`**：
- 如果约束/索引已存在就跳过，不会报错。适合写在脚本里反复执行

> **最佳实践**：在批量导入数据**之前**先创建约束和索引。约束能在导入时拦截脏数据，索引能加速 MERGE 中的 MATCH 查找。本章为了教学顺序才放在后面。

In [ ]:
# 验证唯一约束：尝试插入重复的论文
try:
    driver.execute_query(
        "CREATE (p:Paper {title: 'Attention Is All You Need', year: 2017})"
    )
    print("插入成功（不应该走到这里）")
except Exception as e:
    print(f"✓ 唯一约束生效，插入被拒绝：")
    print(f"  {type(e).__name__}: {str(e)[:100]}...")

### 常见问题

- **MERGE 和 UNIQUE 约束的关系**：`MERGE` 在找不到匹配节点时会 `CREATE`，如果没有唯一约束，并发导入可能在 MATCH 和 CREATE 之间产生竞态条件导致重复。加了唯一约束后，即使并发也不会重复。
- **索引太多会怎样？** 索引会占用额外存储空间，并且写入时需要同步更新索引。只对**经常用来查询的属性**建索引，不要盲目给每个属性都加。
- **约束创建失败**：如果数据中已经有重复值，创建 UNIQUE 约束会失败。需要先清理重复数据再创建约束。

## 6. 查看图的 Schema

In [ ]:
# 查看图中有哪些节点标签
records, _, _ = driver.execute_query("CALL db.labels()")
print("节点标签:")
for r in records:
    print(f"  :{r['label']}")

# 查看图中有哪些关系类型
records, _, _ = driver.execute_query("CALL db.relationshipTypes()")
print("\n关系类型:")
for r in records:
    print(f"  :{r['relationshipType']}")

In [ ]:
# 统计各标签的节点数量
records, _, _ = driver.execute_query("""
    CALL db.labels() YIELD label
    CALL apoc.cypher.run('MATCH (n:`' + label + '`) RETURN count(n) AS count', {})
    YIELD value
    RETURN label, value.count AS count
""")
print("节点统计:")
for r in records:
    print(f"  :{r['label']} -> {r['count']} 个")

### 刚才发生了什么？

- `db.labels()` 返回图中所有节点标签
- `db.relationshipTypes()` 返回所有关系类型
- 结合 APOC 可以动态统计每种标签的节点数量

这些信息在接手一个陌生的图数据库时非常有用——先看 Schema，再决定怎么查询。

## 7. 对比：向量数据导入 vs 图数据导入

学过 LangChain 的你已经做过向量数据库的导入（文档切分 + embedding + FAISS）。和图数据导入有什么区别？

| | LangChain RAG 导入 | Neo4j 图数据导入 |
|--|--|--|
| **数据表示** | `Document(page_content, metadata)` | `Node(labels, properties)` + `Relationship` |
| **导入方式** | `FAISS.from_documents(docs, embeddings)` | `UNWIND + MERGE` 批量 Cypher |
| **预处理** | 文本切分（chunk） | 实体识别 + 关系提取 |
| **索引** | 向量索引（自动） | 属性索引 + 约束（手动创建） |
| **查询方式** | 语义相似度匹配 | 结构化模式匹配（Cypher） |
| **擅长回答** | “和 X 相关的内容是什么？” | “A 和 B 之间有什么关系？” |

两者并不互斥——**GraphRAG** 就是把两者结合起来：用图存储结构化知识，用向量存储非结构化文本，查询时同时利用两种信息。这也是后续章节会涉及的内容。

## 总结

本章学习了：

- **图数据建模**：实体→节点，关系→边，属性→键值对
- **Python driver 的事务模式**：`execute_write` / `execute_read`，自动重试、读写分离
- **UNWIND 批量导入** vs 逐条写入的性能差异（通常快 10x 以上）
- **完整实战**：构建学术论文知识图谱（Paper、Author、Topic 三类节点 + 三种关系）
- **约束（UNIQUE）和索引**：保证数据质量、加速查询
- **Schema 查看**：`db.labels()`、`db.relationshipTypes()` 快速了解图结构

## 下一章预告

**第四章：LangChain Neo4j 集成** —— 前三章我们手写 Cypher 操作图。下一章引入 `langchain-neo4j`，让 LLM 理解图的结构，自动生成 Cypher 查询（Text-to-Cypher），实现自然语言图问答。

In [ ]:
# 关闭驱动
driver.close()
print("✓ 连接已关闭")